# Project 55 — Local GitHub Repo Reader Agent

**Inspect a local codebase and answer questions about its structure, dependencies, and logic.**

| Component | Choice |
|-----------|--------|
| LLM runtime | Ollama (qwen3:8b) |
| Framework | LangChain |
| Tools | Python filesystem + AST parsing |
| Interface | Jupyter notebook |

## 1 · What You Will Learn

1. How to build **code-aware tools** that extract structure from source files
2. How to use **AST parsing** to list functions, classes, and imports
3. How to answer **repo-level questions** by combining file inspection + LLM reasoning
4. How to build a **code search** tool for finding relevant files
5. How to summarize codebases for onboarding and documentation

## 2 · Why This Project Matters

Understanding an unfamiliar codebase is one of the most time-consuming developer tasks.
A local repo reader agent can quickly answer questions like 'What does this module do?',
'Where is X implemented?', or 'What are the dependencies?'. Running locally means your
proprietary code never leaves your machine.

## 3 · Environment Setup

**Prerequisites:**
- Ollama running with `qwen3:8b`
- A local code repository to analyze (we create a sample one)

In [ ]:
# !pip install -q langchain langchain-ollama

## 4 · Imports and Configuration

In [ ]:
import os
import ast
import json
from pathlib import Path

from langchain_ollama import ChatOllama
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

MODEL = 'qwen3:8b'
llm = ChatOllama(model=MODEL, temperature=0.1)
print(f'LLM ready: {MODEL}')

## 5 · Create a Sample Repository

We create a small but realistic Python project structure to analyze.

In [ ]:
REPO_DIR = Path('sample_repo')
if REPO_DIR.exists():
    import shutil
    shutil.rmtree(REPO_DIR)

# Create project structure
(REPO_DIR / 'src').mkdir(parents=True)
(REPO_DIR / 'tests').mkdir(parents=True)

# Main module
(REPO_DIR / 'src' / '__init__.py').write_text('')

(REPO_DIR / 'src' / 'models.py').write_text('''
"""Data models for the task manager application."""
from dataclasses import dataclass, field
from datetime import datetime
from typing import Optional

@dataclass
class Task:
    title: str
    description: str = ""
    priority: int = 1
    completed: bool = False
    created_at: datetime = field(default_factory=datetime.now)
    due_date: Optional[datetime] = None

    def mark_complete(self):
        self.completed = True

    def is_overdue(self) -> bool:
        if self.due_date and not self.completed:
            return datetime.now() > self.due_date
        return False

@dataclass
class Project:
    name: str
    tasks: list = field(default_factory=list)

    def add_task(self, task: Task):
        self.tasks.append(task)

    def completion_rate(self) -> float:
        if not self.tasks:
            return 0.0
        return sum(1 for t in self.tasks if t.completed) / len(self.tasks)
''')

(REPO_DIR / 'src' / 'database.py').write_text('''
"""SQLite database layer for task persistence."""
import sqlite3
from pathlib import Path
from .models import Task, Project

class TaskDatabase:
    def __init__(self, db_path: str = "tasks.db"):
        self.db_path = db_path
        self.conn = sqlite3.connect(db_path)
        self._create_tables()

    def _create_tables(self):
        self.conn.execute("""CREATE TABLE IF NOT EXISTS tasks (
            id INTEGER PRIMARY KEY, title TEXT, description TEXT,
            priority INTEGER, completed BOOLEAN, created_at TEXT, due_date TEXT
        )""")
        self.conn.commit()

    def add_task(self, task: Task) -> int:
        cur = self.conn.execute(
            "INSERT INTO tasks (title, description, priority, completed, created_at) VALUES (?,?,?,?,?)",
            (task.title, task.description, task.priority, task.completed, str(task.created_at))
        )
        self.conn.commit()
        return cur.lastrowid

    def get_all_tasks(self) -> list:
        return self.conn.execute("SELECT * FROM tasks").fetchall()

    def close(self):
        self.conn.close()
''')

(REPO_DIR / 'src' / 'cli.py').write_text('''
"""Command-line interface for the task manager."""
import argparse
from .models import Task
from .database import TaskDatabase

def main():
    parser = argparse.ArgumentParser(description="Task Manager CLI")
    parser.add_argument("action", choices=["add", "list", "complete"])
    parser.add_argument("--title", type=str)
    parser.add_argument("--priority", type=int, default=1)
    args = parser.parse_args()

    db = TaskDatabase()
    if args.action == "add":
        task = Task(title=args.title, priority=args.priority)
        task_id = db.add_task(task)
        print(f"Added task #{task_id}: {args.title}")
    elif args.action == "list":
        tasks = db.get_all_tasks()
        for t in tasks:
            print(t)
    db.close()

if __name__ == "__main__":
    main()
''')

(REPO_DIR / 'tests' / 'test_models.py').write_text('''
"""Tests for data models."""
from src.models import Task, Project

def test_task_creation():
    t = Task(title="Test task")
    assert t.title == "Test task"
    assert not t.completed

def test_mark_complete():
    t = Task(title="Test")
    t.mark_complete()
    assert t.completed

def test_project_completion_rate():
    p = Project(name="Test Project")
    p.add_task(Task(title="A", completed=True))
    p.add_task(Task(title="B", completed=False))
    assert p.completion_rate() == 0.5
''')

(REPO_DIR / 'requirements.txt').write_text('sqlite3\npytest\n')
(REPO_DIR / 'README.md').write_text(
    '# Task Manager\n\nA simple CLI task manager with SQLite storage.\n\n'
    '## Usage\n```bash\npython -m src.cli add --title "My task"\npython -m src.cli list\n```\n'
)

print('Sample repo created at:', REPO_DIR.resolve())
for f in sorted(REPO_DIR.rglob('*')):
    if f.is_file():
        print(f'  {f.relative_to(REPO_DIR)}')

## 6 · Code Analysis Tools

These tools extract structured information from Python files.

In [ ]:
def get_repo_tree(repo_path: Path) -> str:
    """Get the file tree of a repository."""
    lines = []
    for f in sorted(repo_path.rglob('*')):
        if f.is_file() and '__pycache__' not in str(f):
            rel = f.relative_to(repo_path)
            size = f.stat().st_size
            lines.append(f'{rel} ({size} bytes)')
    return '\n'.join(lines)


def read_source_file(repo_path: Path, filepath: str) -> str:
    """Read a source file from the repo."""
    full = repo_path / filepath
    if not full.exists():
        return f'File not found: {filepath}'
    return full.read_text(encoding='utf-8')


def analyze_python_file(repo_path: Path, filepath: str) -> dict:
    """Extract structure from a Python file using AST."""
    full = repo_path / filepath
    if not full.exists() or not filepath.endswith('.py'):
        return {'error': f'Not a valid Python file: {filepath}'}

    source = full.read_text(encoding='utf-8')
    try:
        tree = ast.parse(source)
    except SyntaxError as e:
        return {'error': f'Syntax error: {e}'}

    info = {
        'file': filepath,
        'docstring': ast.get_docstring(tree) or '',
        'imports': [],
        'classes': [],
        'functions': [],
        'line_count': len(source.split('\n')),
    }

    for node in ast.walk(tree):
        if isinstance(node, ast.Import):
            for alias in node.names:
                info['imports'].append(alias.name)
        elif isinstance(node, ast.ImportFrom):
            module = node.module or ''
            for alias in node.names:
                info['imports'].append(f'{module}.{alias.name}')
        elif isinstance(node, ast.ClassDef):
            methods = [n.name for n in node.body if isinstance(n, (ast.FunctionDef, ast.AsyncFunctionDef))]
            info['classes'].append({'name': node.name, 'methods': methods})
        elif isinstance(node, ast.FunctionDef) and isinstance(node, ast.FunctionDef):
            # Only top-level functions
            pass

    # Get top-level functions
    for node in ast.iter_child_nodes(tree):
        if isinstance(node, ast.FunctionDef):
            info['functions'].append(node.name)

    return info


def search_code(repo_path: Path, keyword: str) -> str:
    """Search for a keyword across all source files."""
    results = []
    for f in sorted(repo_path.rglob('*.py')):
        if '__pycache__' in str(f):
            continue
        content = f.read_text(encoding='utf-8')
        for i, line in enumerate(content.split('\n'), 1):
            if keyword.lower() in line.lower():
                rel = f.relative_to(repo_path)
                results.append(f'{rel}:{i}: {line.strip()}')
    return '\n'.join(results) if results else f'No matches for "{keyword}"'


# Test the tools
print('=== Repo Tree ===')
print(get_repo_tree(REPO_DIR))
print('\n=== AST Analysis: src/models.py ===')
info = analyze_python_file(REPO_DIR, 'src/models.py')
print(json.dumps(info, indent=2))

## 7 · Build the Repo Q&A Pipeline

The pipeline gathers context using the tools, then asks the LLM to answer
the user's question about the codebase.

In [ ]:
def ask_about_repo(question: str, repo_path: Path = REPO_DIR, verbose: bool = True) -> dict:
    """Answer a question about a code repository."""
    # Gather context
    tree = get_repo_tree(repo_path)

    # Analyze all Python files
    analyses = []
    for f in sorted(repo_path.rglob('*.py')):
        if '__pycache__' not in str(f):
            rel = str(f.relative_to(repo_path))
            info = analyze_python_file(repo_path, rel)
            analyses.append(info)

    # Search for relevant keywords from the question
    keywords = [w for w in question.lower().split() if len(w) > 3 and w not in
                {'what', 'does', 'this', 'that', 'which', 'where', 'about', 'have', 'with'}]
    search_results = []
    for kw in keywords[:3]:  # limit searches
        sr = search_code(repo_path, kw)
        if not sr.startswith('No matches'):
            search_results.append(f'Search "{kw}":\n{sr}')

    # Read README if exists
    readme = ''
    readme_path = repo_path / 'README.md'
    if readme_path.exists():
        readme = readme_path.read_text(encoding='utf-8')

    # Ask LLM
    prompt = ChatPromptTemplate.from_messages([
        ('system',
         'You are a code analysis assistant. Answer the question about this repository '
         'using the provided context. Be specific, reference file names and function names. '
         'If you cannot determine the answer from the context, say so.'),
        ('human',
         'Question: {question}\n\n'
         'File tree:\n{tree}\n\n'
         'README:\n{readme}\n\n'
         'File analyses:\n{analyses}\n\n'
         'Code search results:\n{search_results}'),
    ])
    chain = prompt | llm | StrOutputParser()
    answer = chain.invoke({
        'question': question,
        'tree': tree,
        'readme': readme,
        'analyses': json.dumps(analyses, indent=1),
        'search_results': '\n\n'.join(search_results) if search_results else 'None',
    })

    if verbose:
        print(f'Q: {question}')
        print(f'\nA: {answer}')

    return {'question': question, 'answer': answer}


print('Repo Q&A pipeline ready.')

## 8 · Ask Questions About the Codebase

In [ ]:
r1 = ask_about_repo('What is the overall architecture of this project?')

In [ ]:
r2 = ask_about_repo('What classes are defined and what do they do?')

In [ ]:
r3 = ask_about_repo('How does the database layer work?')

In [ ]:
r4 = ask_about_repo('What are the external dependencies?')

In [ ]:
r5 = ask_about_repo('Are there any potential bugs or issues in the code?')

## 9 · Generate Onboarding Summary

Produce a comprehensive onboarding document for a new developer.

In [ ]:
# Read all source files
all_source = []
for f in sorted(REPO_DIR.rglob('*.py')):
    if '__pycache__' not in str(f):
        rel = f.relative_to(REPO_DIR)
        content = f.read_text(encoding='utf-8')
        all_source.append(f'--- {rel} ---\n{content}')

onboard_prompt = ChatPromptTemplate.from_messages([
    ('system',
     'You are a senior developer writing onboarding documentation. '
     'Create a clear onboarding guide that covers:\n'
     '1. Project purpose\n'
     '2. Architecture overview\n'
     '3. Key modules and their responsibilities\n'
     '4. How to run the project\n'
     '5. Key patterns and conventions used\n'
     'Reference specific files and functions.'),
    ('human', 'Source files:\n{source}'),
])
chain = onboard_prompt | llm | StrOutputParser()
onboarding = chain.invoke({'source': '\n\n'.join(all_source)})

print('ONBOARDING GUIDE')
print('=' * 60)
print(onboarding)

## 10 · Save Results

In [ ]:
results = {
    'questions': [r1, r2, r3, r4, r5],
    'onboarding': onboarding,
}
with open('repo_reader_results.json', 'w', encoding='utf-8') as f:
    json.dump(results, f, indent=2, ensure_ascii=False)
print('Results saved.')

## 11 · Limitations

- **Python-only AST** — non-Python files are read as plain text
- **Context window** — very large repos need selective file loading
- **No semantic understanding** — the agent reads syntax, not runtime behavior
- **No git history** — doesn't analyze commits or blame
- Searches are keyword-based, not semantic

## 12 · How to Improve

1. **Add embedding-based search** for semantic code search
2. **Parse other languages** (JS, Java) with tree-sitter
3. **Add git log analysis** for change history and authorship
4. **Dependency graph** visualization with graphviz
5. **Auto-generate docstrings** for undocumented functions

## 13 · Mini Challenge

1. Add support for analyzing `requirements.txt` and mapping dependencies
2. Add a `find_todos` tool that finds all TODO/FIXME/HACK comments
3. Generate a test coverage estimate by comparing test files to source files

## 14 · Key Takeaways

| Concept | What You Practiced |
|---------|-------------------|
| AST parsing | Extract structure (classes, functions, imports) from Python |
| Code search | Keyword-based search across source files |
| Contextual Q&A | Feed code context to LLM for repo-level answers |
| Onboarding docs | Auto-generate developer documentation |
| Local-first | All analysis runs locally with Ollama |